# Few-shot fine-tuning on the Bengaluru Solar Panel (BSP) dataset

**Question:** a model trained on Google Earth imagery scores only ~0.30 IoU on Bengaluru zero-shot. How much does fine-tuning on **a few labelled Bengaluru images** fix that? Is starting from the **Google checkpoint** better than starting from **ImageNet only** (what Experiments 3 / 4 did)?

**Protocol**

| | |
|---|---|
| Initialisation | **Google**: the trained crowdsourced checkpoint (`CD_ResnetUnet*.pth`)  vs  **ImageNet**: ImageNet ResNet34 encoder, random decoder |
| Training images (k) | 5, 10 or 20 of the 20 BSP training originals: layout-balanced, nested (the 5 are inside the 10), a different draw per seed |
| Offline augmentation | without: the k originals only;  with: their 8 versions from *BSP augmented* (orig, duplicate, h/v-flip, 3 rotations, blur). k = 20 with augmentation is exactly the 160-image Experiment 4 training set |
| Validation | the 10 BSP val images, used only for early stopping (identical in every run) |
| Test | the 13 BSP test images (identical in every run); also reported per layout |
| Recipe | same as the seeded Bengaluru ablation (`Final Model/UsingAttentions/Attentions.ipynb`): batch 5, AdamW lr 2e-4, weight decay 1e-2 (none on biases / norms), ReduceLROnPlateau(0.7, patience 7, min 1e-6), max 100 epochs, early stopping patience 10 on val loss, grad clip 1.0, loss 0.6 BCE + 0.4 Dice (+ 0.2 x boundary loss for the boundary models), best-val-loss weights evaluated |
| Epoch | **160 training samples for every k** (the k images are repeated, reshuffled each pass), so every setting gets the same number of optimisation steps as the 160-image Experiment 4 |
| Seeds | 0, 1, 2. The seed fixes the image draw, the initialisation of the new layers and the batch order. Results are mean +- std |
| k = 0 | the Google checkpoint evaluated directly (zero-shot), as the starting point of the learning curve |

**Sanity check:** *ImageNet, k = 20 with augmentation, baseline* should reproduce the seeded Experiment-4 baseline, **0.9073 +- 0.0051** test IoU.

**Kaggle setup:** attach the `models` dataset (the six `.pth`), `bangaloresolarpanels` and `bangaloresolarpanels-augmented`. **Turn Internet ON** (ImageNet weights are downloaded from torchvision). Use a GPU. The default grid (2 models x 2 inits x 3 k x 2 aug x 3 seeds = 72 runs) takes several hours, so use **Save Version -> Save & Run All** so it runs in the background. The runtime estimate is printed before training starts; shrink `MODELS_TO_RUN` / `SEEDS` / `SHOTS` if needed.

In [ ]:
import os, re, io, gc, copy, math, time, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}', torch.cuda.get_device_name(0) if device.type == 'cuda' else '')

## 1. Model definitions and losses (identical to the training notebooks)

In [ ]:
# Model definitions copied verbatim from the Google training notebooks
# (CD_UnetResnet.ipynb and CD_ResnetUnet_{AxialTransformer,BoundaryHead,BoundaryGuidedAttention,StyleGate}.ipynb)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_rate=0.15, use_dropout=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if use_dropout:
            layers.append(nn.Dropout2d(dropout_rate))

        layers.extend([
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ])
        if use_dropout:
            layers.append(nn.Dropout2d(dropout_rate))

        self.double_conv = nn.Sequential(*layers)

    def forward(self, x):
        return self.double_conv(x)

# (constructor, ImageNet weights enum name, channels of [stem, layer1, layer2, layer3, layer4])
RESNET_CFG = {
    'resnet18':  (models.resnet18,  'ResNet18_Weights',  [64, 64, 128, 256, 512]),
    'resnet34':  (models.resnet34,  'ResNet34_Weights',  [64, 64, 128, 256, 512]),
    'resnet50':  (models.resnet50,  'ResNet50_Weights',  [64, 256, 512, 1024, 2048]),
    'resnet101': (models.resnet101, 'ResNet101_Weights', [64, 256, 512, 1024, 2048]),
}

class ResNetUNet(nn.Module):
    """U-Net with a ResNet encoder.

    Encoder (strides relative to the input):
        stem   (conv7x7 + BN + ReLU)  /2
        layer1 (after maxpool)        /4
        layer2                        /8
        layer3                        /16
        layer4 (bottleneck)           /32
    Decoder: ConvTranspose2d x2 + skip concat + DoubleConv, five times, back to full resolution.
    Input height/width must be divisible by 32 (256x256 is fine).
    """
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1,
                 dropout_rates=None):
        super().__init__()
        if backbone not in RESNET_CFG:
            raise ValueError(f"backbone must be one of {list(RESNET_CFG)}")

        if dropout_rates is None:
            dropout_rates = {
                'encoder': {'layer3': 0.15, 'layer4': 0.2},   # same idea as U-Net: enc4 = 0.15, bottleneck = 0.2
                'decoder': 0.15,
            }
        self.dropout_rates = dropout_rates

        ctor, weights_name, ch = RESNET_CFG[backbone]
        weights = getattr(models, weights_name).IMAGENET1K_V1 if pretrained else None
        backbone_net = ctor(weights=weights)

        # Optional: support in_ch != 3 by rebuilding the first conv
        if in_ch != 3:
            backbone_net.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Encoder
        self.stem = nn.Sequential(backbone_net.conv1, backbone_net.bn1, backbone_net.relu)  # /2
        self.pool = backbone_net.maxpool                                                    # /4
        self.layer1 = backbone_net.layer1
        self.layer2 = backbone_net.layer2
        self.layer3 = backbone_net.layer3
        self.layer4 = backbone_net.layer4

        self.drop3 = nn.Dropout2d(dropout_rates['encoder']['layer3'])
        self.drop4 = nn.Dropout2d(dropout_rates['encoder']['layer4'])

        c0, c1, c2, c3, c4 = ch
        d = dropout_rates['decoder']

        # Decoder
        self.up4 = nn.ConvTranspose2d(c4, c3, kernel_size=2, stride=2)       # /32 -> /16
        self.dec4 = DoubleConv(c3 * 2, c3, d, use_dropout=True)
        self.up3 = nn.ConvTranspose2d(c3, c2, kernel_size=2, stride=2)       # /16 -> /8
        self.dec3 = DoubleConv(c2 * 2, c2, d, use_dropout=True)
        self.up2 = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)       # /8  -> /4
        self.dec2 = DoubleConv(c1 * 2, c1, d, use_dropout=True)
        self.up1 = nn.ConvTranspose2d(c1, c0, kernel_size=2, stride=2)       # /4  -> /2
        self.dec1 = DoubleConv(c0 * 2, c0, d, use_dropout=True)
        self.up0 = nn.ConvTranspose2d(c0, 32, kernel_size=2, stride=2)       # /2  -> full res
        self.dec0 = DoubleConv(32, 32, d, use_dropout=True)

        self.final_conv = nn.Conv2d(32, out_ch, kernel_size=1)

    def forward(self, x):
        # Encoder
        e0 = self.stem(x)                       # /2
        e1 = self.layer1(self.pool(e0))         # /4
        e2 = self.layer2(e1)                    # /8
        e3 = self.drop3(self.layer3(e2))        # /16
        b = self.drop4(self.layer4(e3))         # /32 (bottleneck)

        # Decoder
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))

        return self.final_conv(d0)

# =====================================================================
# Axial attention (same module as BSP_augmented/Final/Resnet_Axial.ipynb)
# =====================================================================
class AxialAttention2d(nn.Module):
    """Axial self-attention: attend along W (rows), then along H (columns).
    Cost O(HW(H+W)) instead of O((HW)^2). Residual with zero-initialised gate, so at
    init the block is identity and the pretrained ResNet-UNet behaviour is preserved."""
    def __init__(self, dim, heads=None, attn_drop=0.0):
        super().__init__()
        self.heads = heads if heads is not None else max(1, dim // 32)
        assert dim % self.heads == 0
        self.pos = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)       # size-agnostic positional encoding
        self.norm = nn.GroupNorm(1, dim)
        self.qkv_w = nn.Conv2d(dim, dim * 3, 1, bias=False)
        self.qkv_h = nn.Conv2d(dim, dim * 3, 1, bias=False)
        self.proj_w = nn.Conv2d(dim, dim, 1)
        self.proj_h = nn.Conv2d(dim, dim, 1)
        self.attn_drop = attn_drop
        self.gamma = nn.Parameter(torch.zeros(1))

    def _attend(self, x, qkv_layer, proj_layer, axis):
        B, C, H, W = x.shape
        q, k, v = qkv_layer(x).chunk(3, dim=1)
        h = self.heads
        def to_seq(t):                                                  # -> (B*other, heads, L, C/heads)
            if axis == 'w':                                             # sequences run along W
                t = t.permute(0, 2, 3, 1).reshape(B * H, W, h, C // h)
            else:                                                       # sequences run along H
                t = t.permute(0, 3, 2, 1).reshape(B * W, H, h, C // h)
            return t.transpose(1, 2)
        out = F.scaled_dot_product_attention(
            to_seq(q), to_seq(k), to_seq(v),
            dropout_p=self.attn_drop if self.training else 0.0)
        out = out.transpose(1, 2)
        if axis == 'w':
            out = out.reshape(B, H, W, C).permute(0, 3, 1, 2)
        else:
            out = out.reshape(B, W, H, C).permute(0, 3, 2, 1)
        return proj_layer(out.contiguous())

    def forward(self, x):
        y = self.norm(x + self.pos(x))
        y = y + self._attend(y, self.qkv_w, self.proj_w, 'w')          # row attention
        y = y + self._attend(self.norm(y), self.qkv_h, self.proj_h, 'h')  # column attention
        return x + self.gamma * y


class AxialResNetUNet(ResNetUNet):
    """ResNet-UNet with axial attention on the /4, /8 and /16 skip connections."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        _, c1, c2, c3, _ = RESNET_CFG[backbone][2]
        self.ax1, self.ax2, self.ax3 = AxialAttention2d(c1), AxialAttention2d(c2), AxialAttention2d(c3)

    def _enc(self, x):
        e0 = self.stem(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.drop3(self.layer3(e2))
        b = self.drop4(self.layer4(e3))
        return e0, e1, e2, e3, b

    def forward(self, x):
        e0, e1, e2, e3, b = self._enc(x)
        e1, e2, e3 = self.ax1(e1), self.ax2(e2), self.ax3(e3)
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        return self.final_conv(self.dec0(self.up0(d1)))


# =====================================================================
# Transformer on the bottleneck (same module as BSP_augmented/Final/Resnet_AxialTransformer.ipynb)
# =====================================================================
class BottleneckTransformer(nn.Module):
    """Transformer encoder on the /32 bottleneck tokens (8x8 = 64 tokens at 256x256).
    Projects to a small width first so it stays cheap for any ResNet backbone."""
    def __init__(self, in_ch, dim=256, depth=2, heads=8, dropout=0.1):
        super().__init__()
        self.proj_in = nn.Conv2d(in_ch, dim, 1)
        self.pos = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)       # conditional positional encoding
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth, enable_nested_tensor=False)
        self.proj_out = nn.Conv2d(dim, in_ch, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, _, H, W = x.shape
        t = self.proj_in(x)
        t = t + self.pos(t)
        t = self.encoder(t.flatten(2).transpose(1, 2))                  # (B, HW, dim)
        t = t.transpose(1, 2).reshape(B, -1, H, W)
        return x + self.gamma * self.proj_out(t)


class AxialTransResNetUNet(AxialResNetUNet):
    """Axial attention on the /4, /8, /16 skips + 2-layer Transformer on the /32 bottleneck."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1,
                 dropout_rates=None, tr_dim=256, tr_depth=2, tr_heads=8):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        c4 = RESNET_CFG[backbone][2][4]
        self.bottleneck_tr = BottleneckTransformer(c4, tr_dim, tr_depth, tr_heads)

    def forward(self, x):
        e0, e1, e2, e3, b = self._enc(x)
        b = self.bottleneck_tr(b)
        e1, e2, e3 = self.ax1(e1), self.ax2(e2), self.ax3(e3)
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        return self.final_conv(self.dec0(self.up0(d1)))

# =====================================================================
# Auxiliary boundary head (same head as Final Model/ResNet34_BoundaryHead.ipynb)
# =====================================================================
class BoundaryHeadResNetUNet(ResNetUNet):
    """ResNet-UNet + a small boundary head on the full-resolution decoder features.
    The segmentation path is unchanged; the head only adds a training signal.
    forward() returns (segmentation_logits, boundary_logits)."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        self.boundary_head = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, out_ch, kernel_size=1),
        )

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.drop3(self.layer3(e2))
        b = self.drop4(self.layer4(e3))
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))
        return self.final_conv(d0), self.boundary_head(d0)

# =====================================================================
# Boundary-guided attention (same module as Final Model/resnet34_boundaryguidedattention.ipynb)
# =====================================================================
class BoundaryGuidedResNetUNet(ResNetUNet):
    """ResNet-UNet + boundary head whose prediction reweights the full-resolution decoder
    features before the segmentation head:  d0 + scale * d0 * sigmoid(boundary).
    scale starts at 0, so training starts from the plain ResNet-UNet.
    forward() returns (segmentation_logits, boundary_logits)."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        self.boundary_feature = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )
        self.boundary_head = nn.Conv2d(16, out_ch, kernel_size=1)
        self.boundary_attention_scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.drop3(self.layer3(e2))
        b = self.drop4(self.layer4(e3))
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))

        boundary_logits = self.boundary_head(self.boundary_feature(d0))
        boundary_attention = torch.sigmoid(boundary_logits)
        refined = d0 + self.boundary_attention_scale * (d0 * boundary_attention)
        return self.final_conv(refined), boundary_logits

# =====================================================================
# StyleGate (same module as Final Model/UsingAttentions/Attentions.ipynb)
# =====================================================================
class StyleGate(nn.Module):
    '''Style-invariant channel gate:
        out = x + gamma * g * (IN(x) - x),   g = sigmoid(MLP([channel mean, channel std]))  (per channel)
    IN(x) is the instance-normalised feature: the per-image colour / brightness / contrast statistics
    are removed, which is mostly what changes between regions (Google Earth vs Bengaluru).
    g lets every channel decide how much of the style-free version it wants.
    gamma starts at 0 -> exact identity at the start of training; the network learns whether to use it.'''
    def __init__(self, ch, reduction=4):
        super().__init__()
        self.inorm = nn.InstanceNorm2d(ch, affine=False)
        hid = max(ch // reduction, 8)
        self.mlp = nn.Sequential(nn.Linear(2 * ch, hid), nn.ReLU(inplace=True), nn.Linear(hid, ch))
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        mu = x.mean(dim=(2, 3))
        sd = x.var(dim=(2, 3), unbiased=False).add(1e-5).sqrt()
        g = torch.sigmoid(self.mlp(torch.cat([mu, sd], dim=1)))[:, :, None, None]
        return x + self.gamma * g * (self.inorm(x) - x)


class StyleGateResNetUNet(ResNetUNet):
    '''ResNet-UNet with a StyleGate after the stem (/2) and after layer1 (/4),
    i.e. where low-level appearance (colour, texture, contrast) is encoded.'''
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        c0, c1, _, _, _ = RESNET_CFG[backbone][2]
        self.style0 = StyleGate(c0)
        self.style1 = StyleGate(c1)

    def forward(self, x):
        e0 = self.style0(self.stem(x))                  # /2
        e1 = self.style1(self.layer1(self.pool(e0)))    # /4
        e2 = self.layer2(e1)                            # /8
        e3 = self.drop3(self.layer3(e2))                # /16
        b = self.drop4(self.layer4(e3))                 # /32
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        return self.final_conv(self.dec0(self.up0(d1)))

In [ ]:
# Losses copied verbatim from the training notebooks:
#   dice_coef / DiceLoss / combined_loss (0.6 BCE + 0.4 Dice) -> every notebook
#   BOUNDARY_WEIGHT / boundary_target_from_mask / boundary_loss_fn -> boundary notebooks

def dice_coef(pred, target, smooth=1e-6):
    pred = pred.contiguous()
    target = target.contiguous()
    intersection = (pred * target).sum(dim=(2,3))
    denom = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    dice = (2. * intersection + smooth) / (denom + smooth)
    return dice.mean()


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        dice = dice_coef(pred, target, smooth=self.smooth)
        return 1.0 - dice


def combined_loss(logits, mask, bce_weight=0.6, dice_weight=0.4):
    bce_loss = nn.BCEWithLogitsLoss()
    bce = bce_loss(logits, mask)
    probs = torch.sigmoid(logits)
    dice = DiceLoss()(probs, mask)
    return bce_weight * bce + dice_weight * dice


BOUNDARY_WEIGHT = 0.2


def boundary_target_from_mask(mask):
    """~2 px wide boundary band (dilation minus erosion with a 3x3 window) from a binary mask."""
    dilated = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)
    eroded = 1.0 - F.max_pool2d(1.0 - mask, kernel_size=3, stride=1, padding=1)
    return (dilated - eroded).clamp(0.0, 1.0)


def boundary_loss_fn(boundary_logits, mask):
    """0.5 * class-balanced BCE + 0.5 * Dice on the boundary band."""
    target = boundary_target_from_mask(mask)
    positives = target.sum()
    negatives = target.numel() - positives
    pos_weight = (negatives / positives.clamp_min(1.0)).clamp(min=1.0, max=20.0).reshape(1)
    bce = F.binary_cross_entropy_with_logits(boundary_logits, target, pos_weight=pos_weight)
    return 0.5 * bce + 0.5 * DiceLoss()(torch.sigmoid(boundary_logits), target)

## 2. Configuration
Paths default to the `dewangchoudhary` Kaggle datasets. If a path does not exist, it is searched for automatically under `/kaggle/input`.

In [ ]:
BSP_ROOT = '/kaggle/input/datasets/dewangchoudhary/bangaloresolarpanels/BangaloreSolarPanels'
AUG_ROOT = '/kaggle/input/datasets/dewangchoudhary/bangaloresolarpanels-augmented/BangaloreSolarPanels_augmented'
CKPT_DIR = '/kaggle/input/datasets/dewangchoudhary/models/Models'
OUT_DIR = '/kaggle/working/fewshot'
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- experiment grid ----------------
MODELS_TO_RUN = ['ResNet-UNet (baseline)', 'Axial']   # any names from MODEL_SPECS below
INITS = ['Google', 'ImageNet']
SHOTS = [5, 10, 20]                 # number of BSP training originals
USE_AUG = [False, True]             # False: k originals; True: their 8 offline-augmented versions
SEEDS = [0, 1, 2]

# ---------------- training recipe (same as the seeded Bengaluru ablation) ----------------
CFG = dict(batch_size=5, lr=2e-4, weight_decay=1e-2, max_epochs=100, patience=10, min_delta=1e-4,
           sched_factor=0.7, sched_patience=7, min_lr=1e-6, grad_clip=1.0, samples_per_epoch=160)
FREEZE_ENCODER = False              # True: train only decoder (+ new modules); encoder kept from the init
IMG_SIZE = (256, 256)
THRESH = 0.5
DROPOUT_CONFIG = {'encoder': {'layer3': 0.15, 'layer4': 0.2}, 'decoder': 0.15}

SAVE_CKPTS = False                  # True: save every fine-tuned model (~100 MB each)
PREVIOUS_RESULTS = None             # path to an earlier fewshot_runs.csv: runs already in it are skipped

# (name, class, Google checkpoint, has boundary output)
MODEL_SPECS = {
    'ResNet-UNet (baseline)':    (ResNetUNet,               'CD_ResnetUnet.pth',                         False),
    'Axial':                     (AxialResNetUNet,          'CD_ResnetUnet_Axial.pth',                   False),
    'Axial + Transformer':       (AxialTransResNetUNet,     'CD_ResnetUnet_AxialTransformer.pth',        False),
    'Boundary Head':             (BoundaryHeadResNetUNet,   'CD_ResnetUnet_BoundaryHead.pth',            True),
    'Boundary-Guided Attention': (BoundaryGuidedResNetUNet, 'CD_ResnetUnet_BoundaryGuidedAttention.pth', True),
    'StyleGate':                 (StyleGateResNetUNet,      'CD_ResnetUnet_StyleGate.pth',               False),
}
assert all(m in MODEL_SPECS for m in MODELS_TO_RUN), MODELS_TO_RUN

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')


def ci_sub(parent, name):
    if parent is None or not os.path.isdir(parent):
        return None
    for d in os.listdir(parent):
        if d.lower() == name.lower() and os.path.isdir(os.path.join(parent, d)):
            return os.path.join(parent, d)
    return None


def split_dirs(root):
    out = {}
    for s in ('train', 'val', 'test'):
        sd = ci_sub(root, s)
        im, mk = ci_sub(sd, 'img'), ci_sub(sd, 'masks')
        if im is None or mk is None:
            return None
        out[s] = (im, mk)
    return out


def n_train(root):
    return sum(f.lower().endswith(IMG_EXTS) for f in os.listdir(split_dirs(root)['train'][0]))


if not (BSP_ROOT and split_dirs(BSP_ROOT)) or not (AUG_ROOT and split_dirs(AUG_ROOT)) or not os.path.isdir(CKPT_DIR or ''):
    print('Some default paths do not exist - searching /kaggle/input ...')
    dirs, pths = [], {}
    for dp, _, fns in os.walk('/kaggle/input', followlinks=True):
        dirs.append(dp)
        for f in fns:
            if f.endswith('.pth'):
                pths.setdefault(f, dp)
    cands = [d for d in dirs if split_dirs(d)]
    if not (BSP_ROOT and split_dirs(BSP_ROOT)):
        BSP_ROOT = min([d for d in cands if n_train(d) < 60], key=lambda d: abs(n_train(d) - 20))
    if not (AUG_ROOT and split_dirs(AUG_ROOT)):
        AUG_ROOT = min([d for d in cands if n_train(d) >= 60], key=lambda d: abs(n_train(d) - 160))
    if not os.path.isdir(CKPT_DIR or ''):
        CKPT_DIR = pths.get('CD_ResnetUnet.pth') or next(iter(pths.values()))
print('BSP_ROOT =', BSP_ROOT, f'(train {n_train(BSP_ROOT)})')
print('AUG_ROOT =', AUG_ROOT, f'(train {n_train(AUG_ROOT)})')
print('CKPT_DIR =', CKPT_DIR)
for m in MODELS_TO_RUN:
    p = os.path.join(CKPT_DIR, MODEL_SPECS[m][1])
    print(f'  {m:27s} {"OK" if os.path.isfile(p) else "MISSING (Google init will be skipped)"}  {p}')

## 3. Load the data
Same preprocessing as all training notebooks: resize to 256x256, ImageNet normalisation, mask = pixel > 0. Everything is kept in memory (it is small).

In [ ]:
NORM = transforms.Compose([transforms.ToTensor(),
                           transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
AUG_TAGS = ('orig', 'hflip', 'vflip', 'rotA', 'rotB', 'rotC', 'blur')


def layout_of(stem):
    s = stem.lower()
    if s.startswith('behind_layout') or s.startswith('haralur'):
        return 'Haralur'
    if s.startswith('hsr'):
        return 'HSR'
    if s.startswith('kristal') or s.startswith('krystal'):
        return 'Krystal'
    return 'Other'


def load_split(root, split):
    img_dir, mask_dir = split_dirs(root)[split]
    masks = {}
    for f in os.listdir(mask_dir):
        if f.lower().endswith(IMG_EXTS):
            st = os.path.splitext(f)[0]
            masks[st[:-5] if st.lower().endswith('_mask') else st] = f
    recs = []
    for f in sorted(os.listdir(img_dir)):
        st = os.path.splitext(f)[0]
        if not f.lower().endswith(IMG_EXTS) or st not in masks:
            continue
        m = re.search(r'_(' + '|'.join(AUG_TAGS) + r')$', st)
        aug, source = (m.group(1), st[:m.start()]) if m else ('original', st)
        img = Image.open(os.path.join(img_dir, f)).convert('RGB').resize(IMG_SIZE)
        lab = Image.open(os.path.join(mask_dir, masks[st])).convert('L').resize(IMG_SIZE, Image.NEAREST)
        recs.append(dict(name=st, layout=layout_of(st), aug=aug, source=source, rgb=np.array(img),
                         x=NORM(img), y=torch.from_numpy((np.array(lab) > 0).astype(np.float32))))
    return recs


bsp = {s: load_split(BSP_ROOT, s) for s in ('train', 'val', 'test')}
aug_train = load_split(AUG_ROOT, 'train')
print({s: len(v) for s, v in bsp.items()}, '| augmented train:', len(aug_train))

X_val = torch.stack([r['x'] for r in bsp['val']]);  Y_val = torch.stack([r['y'] for r in bsp['val']]).unsqueeze(1)
X_test = torch.stack([r['x'] for r in bsp['test']]); Y_test = torch.stack([r['y'] for r in bsp['test']]).unsqueeze(1)
test_meta = pd.DataFrame([{k: r[k] for k in ('name', 'layout')} for r in bsp['test']])
TEST_LAYOUT_IDX = {lay: np.where(test_meta.layout == lay)[0] for lay in ('Haralur', 'HSR', 'Krystal')}

train_names = [r['name'] for r in bsp['train']]
train_layout = {r['name']: r['layout'] for r in bsp['train']}
aug_by_source = {}
for i, r in enumerate(aug_train):
    aug_by_source.setdefault(r['source'], []).append(i)
missing = [n for n in train_names if len(aug_by_source.get(n, [])) != 8]
print('augmented versions per training original:', {len(v) for v in aug_by_source.values()},
      '| originals without 8 versions:', missing or 'none')
print('train layouts:', pd.Series(train_layout).value_counts().to_dict(), '| test layouts:', test_meta.layout.value_counts().to_dict())

## 4. Few-shot training sets
For each seed, the 20 training originals are put in a random layout-balanced order (round-robin over Haralur / HSR / Krystal), and the first k are used. Therefore:
* the 5-shot set is inside the 10-shot set, which is inside the 20;
* both inits and both aug settings use **the same images** for a given seed and k, so they can be compared pairwise.

In [ ]:
def shot_order(seed):
    rng = np.random.RandomState(seed)
    by_lay = {}
    for n in train_names:
        by_lay.setdefault(train_layout[n], []).append(n)
    lays = sorted(by_lay)
    for l in lays:
        rng.shuffle(by_lay[l])
    rng.shuffle(lays)
    order = []
    while any(by_lay[l] for l in lays):
        for l in lays:
            if by_lay[l]:
                order.append(by_lay[l].pop(0))
    return order


def train_set(k, use_aug, seed):
    names = shot_order(seed)[:k]
    if use_aug:
        recs = [aug_train[i] for n in names for i in aug_by_source[n]]
    else:
        recs = [r for r in bsp['train'] if r['name'] in names]
    X = torch.stack([r['x'] for r in recs])
    Y = torch.stack([r['y'] for r in recs]).unsqueeze(1)
    return X, Y, names


rows = []
for seed in SEEDS:
    order = shot_order(seed)
    for k in SHOTS:
        rows.append({'seed': seed, 'k': k, 'layouts': pd.Series([train_layout[n] for n in order[:k]]).value_counts().to_dict(),
                     'images': ', '.join(order[:k])})
display(pd.DataFrame(rows))

## 5. Helpers: model construction, evaluation, one fine-tuning run

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_model(name, init):
    cls, ckpt, _ = MODEL_SPECS[name]
    if init == 'Google':
        m = cls(backbone='resnet34', pretrained=False, in_ch=3, out_ch=1, dropout_rates=DROPOUT_CONFIG)
        sd = torch.load(os.path.join(CKPT_DIR, ckpt), map_location='cpu')
        m.load_state_dict({(k[7:] if k.startswith('module.') else k): v for k, v in sd.items()}, strict=True)
    elif init == 'ImageNet':
        m = cls(backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=DROPOUT_CONFIG)
    else:
        raise ValueError(init)
    if FREEZE_ENCODER:
        for n, p in m.named_parameters():
            if n.split('.')[0] in ('stem', 'layer1', 'layer2', 'layer3', 'layer4'):
                p.requires_grad = False
    return m.to(device)


def seg_out(out):
    return out[0] if isinstance(out, (tuple, list)) else out


@torch.no_grad()
def predict(model, X, bs=8):
    model.eval()
    return torch.cat([torch.sigmoid(seg_out(model(X[i:i + bs].to(device)))).cpu() for i in range(0, len(X), bs)])


@torch.no_grad()
def val_loss(model, X, Y, bs=2):
    # segmentation loss only (as in all notebooks: used for the scheduler, early stopping and model choice)
    model.eval()
    tot = 0.0
    for i in range(0, len(X), bs):
        xb, yb = X[i:i + bs].to(device), Y[i:i + bs].to(device)
        tot += combined_loss(seg_out(model(xb)), yb).item() * len(xb)
    return tot / len(X)


def metrics(P, Y, idx=None, thr=THRESH):
    if idx is not None:
        P, Y = P[idx], Y[idx]
    pred, gt = P >= thr, Y > 0.5
    tp = (pred & gt).sum().item(); fp = (pred & ~gt).sum().item()
    fn = (~pred & gt).sum().item(); tn = (~pred & ~gt).sum().item()
    eps = 1e-8
    iou = tp / (tp + fp + fn + eps)
    prec, rec = tp / (tp + fp + eps), tp / (tp + fn + eps)
    return {'IoU': iou, 'mIoU': (iou + tn / (tn + fp + fn + eps)) / 2, 'Dice': 2 * tp / (2 * tp + fp + fn + eps),
            'Precision': prec, 'Recall': rec, 'F1': 2 * prec * rec / (prec + rec + eps)}


def evaluate(model):
    P_test = predict(model, X_test)
    P_val = predict(model, X_val)
    row = {f'test_{k}': v for k, v in metrics(P_test, Y_test).items()}
    row['val_IoU'] = metrics(P_val, Y_val)['IoU']
    for lay, idx in TEST_LAYOUT_IDX.items():
        row[f'test_IoU_{lay}'] = metrics(P_test, Y_test, idx)['IoU']
    return row, P_test


def fine_tune(name, init, k, use_aug, seed):
    set_seed(seed)
    Xtr, Ytr, names = train_set(k, use_aug, seed)
    model = make_model(name, init)
    has_boundary = MODEL_SPECS[name][2]

    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if p.requires_grad:
            (no_decay if (p.ndim == 1 or n.endswith('.bias')) else decay).append(p)
    opt = optim.AdamW([{'params': decay, 'weight_decay': CFG['weight_decay']},
                       {'params': no_decay, 'weight_decay': 0.0}], lr=CFG['lr'])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=CFG['sched_factor'],
                                                 patience=CFG['sched_patience'], min_lr=CFG['min_lr'])
    g = torch.Generator(); g.manual_seed(seed)
    ds = TensorDataset(Xtr, Ytr)
    loader = DataLoader(ds, batch_size=CFG['batch_size'], num_workers=0,
                        sampler=RandomSampler(ds, replacement=False, num_samples=CFG['samples_per_epoch'], generator=g))

    best, best_state, best_epoch, counter, t0 = None, None, 0, 0, time.time()
    for epoch in range(1, CFG['max_epochs'] + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            out = model(xb)
            if has_boundary:
                loss = combined_loss(out[0], yb) + BOUNDARY_WEIGHT * boundary_loss_fn(out[1], yb)
            else:
                loss = combined_loss(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG['grad_clip'])
            opt.step()
        vl = val_loss(model, X_val, Y_val)
        sched.step(vl)
        if best is None or best - vl > CFG['min_delta']:
            best, best_state, best_epoch, counter = vl, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            counter += 1
            if counter >= CFG['patience']:
                break
    model.load_state_dict(best_state)          # best-val-loss weights, as in all notebooks
    row, P_test = evaluate(model)
    row.update({'model': name, 'init': init, 'k': k, 'aug': use_aug, 'seed': seed,
                'n_train_images': len(Xtr), 'train_images': ' '.join(names),
                'epochs_run': epoch, 'best_epoch': best_epoch, 'best_val_loss': best,
                'minutes': (time.time() - t0) / 60})
    if SAVE_CKPTS:
        torch.save(best_state, f"{OUT_DIR}/ft_{name.replace(' ', '')}_{init}_k{k}_{'aug' if use_aug else 'noaug'}_s{seed}.pth")
    del model, opt, best_state
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return row, P_test

## 6. k = 0: the Google checkpoints without fine-tuning (start of the learning curve)
These should match the zero-shot evaluation notebook on the 13 test images: baseline **0.3559**, Axial **0.4186** IoU.

In [ ]:
zero_rows, P_store = [], {}
for name in MODELS_TO_RUN:
    if not os.path.isfile(os.path.join(CKPT_DIR, MODEL_SPECS[name][1])):
        continue
    m = make_model(name, 'Google')
    row, P = evaluate(m)
    row.update({'model': name, 'init': 'Google', 'k': 0, 'aug': False, 'seed': -1, 'n_train_images': 0})
    zero_rows.append(row)
    P_store[(name, 'Google', 0, False)] = P
    del m
df_zero = pd.DataFrame(zero_rows)
display(df_zero[['model', 'test_IoU', 'test_F1', 'test_Precision', 'test_Recall', 'val_IoU',
                 'test_IoU_Haralur', 'test_IoU_HSR', 'test_IoU_Krystal']].set_index('model'))

## 7. Run the grid
One line is printed per run, and results are appended to `fewshot_runs.csv` after every run, so a stopped session loses nothing that finished. To resume in a new session, attach the old CSV as input and set `PREVIOUS_RESULTS`.

In [ ]:
RUNS_CSV = f'{OUT_DIR}/fewshot_runs.csv'
plan = [(m, i, k, a, s) for m in MODELS_TO_RUN for i in INITS for k in SHOTS for a in USE_AUG for s in SEEDS
        if not (i == 'Google' and not os.path.isfile(os.path.join(CKPT_DIR, MODEL_SPECS[m][1])))]

done = pd.DataFrame()
if PREVIOUS_RESULTS and os.path.isfile(PREVIOUS_RESULTS):
    done = pd.read_csv(PREVIOUS_RESULTS)
    done.to_csv(RUNS_CSV, index=False)
done_keys = set(zip(done.model, done.init, done.k, done.aug, done.seed)) if len(done) else set()
todo = [p for p in plan if p not in done_keys]
print(f'{len(plan)} runs in the grid, {len(plan) - len(todo)} already done, {len(todo)} to run.')
print('Rough estimate: ~3-6 min per run on a P100/T4 ->', f'{len(todo) * 3 / 60:.1f}-{len(todo) * 6 / 60:.1f} h')

run_rows = done.to_dict('records')
for j, (name, init, k, use_aug, seed) in enumerate(todo, 1):
    row, P = fine_tune(name, init, k, use_aug, seed)
    run_rows.append(row)
    if seed == SEEDS[0]:
        P_store[(name, init, k, use_aug)] = P
    pd.DataFrame(run_rows).to_csv(RUNS_CSV, index=False)
    print(f"[{j:3d}/{len(todo)}] {name:27s} {init:8s} k={k:2d} aug={str(use_aug):5s} seed={seed} | "
          f"{row['n_train_images']:3d} imgs | epochs {row['epochs_run']:3d} (best {row['best_epoch']:3d}) | "
          f"val IoU {row['val_IoU']:.4f} | test IoU {row['test_IoU']:.4f} F1 {row['test_F1']:.4f} | {row['minutes']:.1f} min")

runs = pd.DataFrame(run_rows)
print('\nSaved', RUNS_CSV)

## 8. Results: test IoU and F1, mean +- std over seeds
Column `k=0` is the Google checkpoint without fine-tuning. `+aug` means the k originals were replaced by their 8 offline-augmented versions (8k images).

In [ ]:
def col_name(k, a):
    return 'k=0 (zero-shot)' if k == 0 else f"k={k}{' +aug' if a else ''}"

allr = pd.concat([df_zero.assign(seed=-1), runs], ignore_index=True)
allr['setting'] = [col_name(k, a) for k, a in zip(allr.k, allr.aug)]
COLS = ['k=0 (zero-shot)'] + [col_name(k, a) for a in USE_AUG for k in SHOTS]

summary = allr.groupby(['model', 'init', 'setting']).agg(
    IoU_mean=('test_IoU', 'mean'), IoU_std=('test_IoU', 'std'), F1_mean=('test_F1', 'mean'), F1_std=('test_F1', 'std'),
    P_mean=('test_Precision', 'mean'), R_mean=('test_Recall', 'mean'), n=('test_IoU', 'size')).reset_index()
summary.to_csv(f'{OUT_DIR}/fewshot_summary.csv', index=False)


def pm(metric):
    t = summary.copy()
    t['v'] = [f'{m:.4f}' if (n == 1 or np.isnan(s)) else f'{m:.4f} ± {s:.4f}'
              for m, s, n in zip(t[f'{metric}_mean'], t[f'{metric}_std'], t['n'])]
    p = t.pivot_table(index=['model', 'init'], columns='setting', values='v', aggfunc='first')
    return p[[c for c in COLS if c in p.columns]].reindex([(m, i) for m in MODELS_TO_RUN for i in INITS]).fillna('')

print('Test IoU (13 BSP test images), mean ± std over seeds')
display(pm('IoU'))
print('Test F1 (= Dice), mean ± std over seeds')
display(pm('F1'))

ref = summary[(summary.model == 'ResNet-UNet (baseline)') & (summary.init == 'ImageNet') & (summary.setting == 'k=20 +aug')]
if len(ref):
    print(f"Sanity check - ImageNet baseline, k=20 +aug: {ref.IoU_mean.iloc[0]:.4f} ± {ref.IoU_std.iloc[0]:.4f}"
          '   (seeded Experiment-4 baseline: 0.9073 ± 0.0051)')

## 9. Learning curves
Solid line = Google initialisation, dashed = ImageNet. Error bars = std over seeds.

In [ ]:
fig, axes = plt.subplots(1, len(USE_AUG), figsize=(7.5 * len(USE_AUG), 5), squeeze=False, sharey=True)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
for c, a in enumerate(USE_AUG):
    ax = axes[0, c]
    for mi, name in enumerate(MODELS_TO_RUN):
        for init in INITS:
            pts = []
            if init == 'Google':
                z = summary[(summary.model == name) & (summary.init == 'Google') & (summary.setting == 'k=0 (zero-shot)')]
                if len(z):
                    pts.append((0, z.IoU_mean.iloc[0], 0.0))
            for k in SHOTS:
                s = summary[(summary.model == name) & (summary.init == init) & (summary.setting == col_name(k, a))]
                if len(s):
                    pts.append((k, s.IoU_mean.iloc[0], 0.0 if np.isnan(s.IoU_std.iloc[0]) else s.IoU_std.iloc[0]))
            if pts:
                x, y, e = zip(*pts)
                ax.errorbar(x, y, yerr=e, marker='o', capsize=3, color=colors[mi % len(colors)],
                            ls='-' if init == 'Google' else '--', label=f'{name} / {init}')
    ax.set_xticks([0] + SHOTS)
    ax.set_xlabel('number of labelled BSP training images (k)' + (' (each x8 augmented)' if a else ''))
    ax.set_title(f"Test IoU vs k - {'with' if a else 'without'} offline augmentation")
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel('test IoU (13 BSP test images)')
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fewshot_learning_curves.png', dpi=130, bbox_inches='tight'); plt.show()

## 10. Does Google pretraining help? (paired: same images and seed, only the initialisation differs)
Positive = the Google checkpoint is better than ImageNet. `wins` counts the seeds where Google was better.

In [ ]:
if set(INITS) >= {'Google', 'ImageNet'}:
    key = ['model', 'k', 'aug', 'seed']
    g = runs[runs.init == 'Google'].set_index(key)
    i = runs[runs.init == 'ImageNet'].set_index(key)
    both = g[['test_IoU', 'test_F1']].join(i[['test_IoU', 'test_F1']], lsuffix='_G', rsuffix='_I', how='inner')
    both['dIoU'] = both.test_IoU_G - both.test_IoU_I
    both['dF1'] = both.test_F1_G - both.test_F1_I
    gain = both.reset_index().groupby(['model', 'k', 'aug']).agg(
        IoU_Google=('test_IoU_G', 'mean'), IoU_ImageNet=('test_IoU_I', 'mean'),
        dIoU_mean=('dIoU', 'mean'), dIoU_std=('dIoU', 'std'), dF1_mean=('dF1', 'mean'),
        wins=('dIoU', lambda s: f'{int((s > 0).sum())}/{len(s)}'))
    gain.to_csv(f'{OUT_DIR}/google_vs_imagenet_gain.csv')
    display(gain)

## 11. Per-layout test IoU (mean over seeds)

In [ ]:
lay = allr.groupby(['model', 'init', 'setting'])[['test_IoU_Haralur', 'test_IoU_HSR', 'test_IoU_Krystal']].mean()
lay.columns = ['Haralur', 'HSR', 'Krystal']
lay = lay.reset_index()
lay['setting'] = pd.Categorical(lay.setting, [c for c in COLS if c in set(lay.setting)], ordered=True)
lay = lay.sort_values(['model', 'init', 'setting']).set_index(['model', 'init', 'setting'])
lay.to_csv(f'{OUT_DIR}/fewshot_per_layout.csv')
display(lay)

## 12. Overlays on the 13 test images (seed 0)
**green = correct panel pixels**, **red = false positive**, **blue = missed panel**. Columns: each model x init for the chosen setting.

In [ ]:
TEST_RGB = np.stack([r['rgb'] for r in bsp['test']])
OVERLAY_SETTINGS = [(5, False), (20, True)]      # (k, aug) to visualise


def overlay(rgb, pred, gt):
    out = rgb.astype(np.float32) / 255.0 * 0.55
    for msk, c in ((pred & gt, (0.1, 0.95, 0.1)), (pred & ~gt, (1.0, 0.15, 0.15)), (~pred & gt, (0.2, 0.45, 1.0))):
        out[msk] = 0.35 * out[msk] + 0.65 * np.array(c)
    return np.clip(out, 0, 1)


for k, a in OVERLAY_SETTINGS:
    keys = [(m, i, k, a) for m in MODELS_TO_RUN for i in INITS if (m, i, k, a) in P_store]
    if not keys:
        continue
    cols = 2 + len(keys)
    fig, axes = plt.subplots(len(TEST_RGB), cols, figsize=(2.4 * cols, 2.5 * len(TEST_RGB)), squeeze=False)
    for r in range(len(TEST_RGB)):
        gt = Y_test[r, 0].numpy() > 0.5
        axes[r, 0].imshow(TEST_RGB[r]); axes[r, 0].set_title(test_meta.name.iloc[r], fontsize=7)
        axes[r, 1].imshow(gt, cmap='gray'); axes[r, 1].set_title('ground truth', fontsize=8)
        for c, key in enumerate(keys):
            pred = P_store[key][r, 0].numpy() >= THRESH
            iou = (pred & gt).sum() / max((pred | gt).sum(), 1)
            axes[r, c + 2].imshow(overlay(TEST_RGB[r], pred, gt))
            axes[r, c + 2].set_title(f'{key[0]}\n{key[1]} | IoU {iou:.2f}', fontsize=7)
        for x in axes[r]:
            x.axis('off')
    fig.suptitle(f"Test predictions after fine-tuning with k={k}{' +aug' if a else ''} (seed {SEEDS[0]})", fontsize=12, y=1.0)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/overlay_k{k}{'_aug' if a else ''}.png", dpi=100, bbox_inches='tight'); plt.show()

## 13. Saved outputs

In [ ]:
for f in sorted(os.listdir(OUT_DIR)):
    print(f'{OUT_DIR}/{f}  ({os.path.getsize(os.path.join(OUT_DIR, f)) / 1024:.0f} KB)')